# Managed-memory union — layer ladder on CACTUS (dev split first)

This notebook has not been executed and contains no result.

Section 0 runs the frozen three-arm pipeline (*prior*) only on the dev split of a **new** CACTUS roster, because the earlier 600-case run directory was not kept. Everything is written to Google Drive. The union run then reuses that prior: its roster, checked memory items and no-RAG, raw-RAG and frozen managed answers, and only generates **new managed-arm answers**, one per layer config:

| Config | Layers |
| --- | --- |
| `L1_raw_plus_items` | raw-RAG whole messages + ranked items |
| `L2_plus_rerank` | + re-rank by recency, importance, reliability |
| `L3_plus_fact_key` | + up to 3 extra messages through the fact-key channel |
| `L4_plus_pinned_profile` | + pinned profile, v5 renderer |

Order of work: **dev split only** → read the comparison → freeze configs → prepare and run the **test** split once. Paid steps are off by default. The API key is read from the existing Colab Secret `Capstone` and never written.

In [ ]:
import os, sys, json, subprocess, importlib.util
from pathlib import Path
if not (Path('/content').is_dir() and os.environ.get('COLAB_RELEASE_TAG') and importlib.util.find_spec('google.colab') is not None):
    raise RuntimeError('Genuine hosted Colab required; do not connect a local runtime.')
print('hosted Colab, python', sys.version.split()[0])

In [ ]:
CODE = 'ad78576df3cbd8f3c0d3df8c4616d0847e1f918d'  # branch feat/managed-union-layers
REPO = Path('/content/cbt-union-' + CODE[:12])
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--quiet', '--no-checkout', 'https://github.com/nanlalala/cbt-memory-system.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--quiet', 'origin', 'feat/managed-union-layers'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--quiet', '--detach', CODE], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == CODE
assert subprocess.run(['git', '-C', str(REPO), 'diff', '--quiet', 'HEAD']).returncode == 0
if importlib.util.find_spec('sentence_transformers') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'sentence-transformers'], check=True)
sys.path.insert(0, str(REPO / 'evaluation/cbt-memory-effect'))
from architecture_retest import runner, judge
from architecture_retest.transport import HostedGatewayTransport
from architecture_union import run as union_run
print('code', CODE)

## Paths
All folders are on Google Drive so nothing is lost when the runtime ends. New folders are created by the code; an existing folder is reused only by the same code version.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/cbt-union-runs')   # everything persists here
WORK.mkdir(parents=True, exist_ok=True)
SPLIT = 'dev'          # change to 'test' only after configs are frozen
DEV_FAMILIES = 40
SOURCE = WORK / 'cactus.json'                       # verified public CACTUS file
PRIOR = WORK / ('prior-cactus-' + CODE[:8])          # fresh frozen three-arm run
UNION = WORK / ('union-' + SPLIT + '-' + CODE[:8])
QUALITY = WORK / ('quality-' + SPLIT + '-' + CODE[:8])
print(json.dumps({'prior': str(PRIOR), 'union': str(UNION), 'quality': str(QUALITY)}, indent=1))
JUDGE_IDENTITY = {'provider': 'https://tokendance.space/gateway/v1', 'model': 'qwen3.5-plus',
                  'returned_model': 'qwen3.5-plus-2026-04-20', 'model_family': 'qwen'}

In [ ]:
from contextlib import contextmanager
@contextmanager
def api_key():
    from google.colab import userdata
    secret = userdata.get('Capstone')
    if not isinstance(secret, str) or not secret.strip():
        raise RuntimeError('Colab Secret Capstone unavailable; nothing sent.')
    try:
        yield secret
    finally:
        secret = None
print('key helper ready; no Secret read yet')

## 0. Fresh prior: frozen three-arm pipeline on the dev split
Downloads the public CACTUS file at the fixed revision and checks its size and SHA-256. Without the historical family list, the public deterministic compiler selects 200 source families, so this roster can differ from the earlier 600-case run: report it as a new, independent run. Only the dev families are run: per case one extraction, one support check and three answers (≈ 5 calls; about 600 for 120 cases). The independent pre-answer assessor is not run; R1–R5 judging is done later in one batch.

In [ ]:
import cactus_materials
if not SOURCE.exists():
    cactus_materials.download_cactus(SOURCE)
FAMILY_IDS = None   # set to the historical 200-family list only if its frozen copy still exists
if not (PRIOR / 'protocol.json').exists():
    p = union_run.prepare_prior(SOURCE, PRIOR, family_ids=FAMILY_IDS)
    print('roster frozen:', p['case_count'], 'cases,', p['source_families'], 'families')

RUN_PRIOR = False
PRIOR_MAX_CALLS = 20      # first batch small; then e.g. 150 per batch
if RUN_PRIOR:
    with api_key() as key:
        print(json.dumps(union_run.run_prior(PRIOR, HostedGatewayTransport(runner.IDENTITY, key),
                                             max_new_calls=PRIOR_MAX_CALLS, split=SPLIT,
                                             dev_families=DEV_FAMILIES, workers=4), indent=1))
else:
    print('Paid prior run disabled.')
    if (PRIOR / ('union_prior_summary_' + SPLIT + '.json')).exists():
        print((PRIOR / ('union_prior_summary_' + SPLIT + '.json')).read_text())

## 0b. Optional pilot on cases whose prior run already finished
For an early look while section 0 is still running. **Exploratory only**: few cases, no confidence claims, never a substitute for the full dev run.

First a zero-cost diagnosis (loads E5, no model call): for each config, is the raw selection inside, how many items fit compared with the frozen managed arm, how many fact-key messages, and request size against the 24,000-character ceiling. Then, if switched on, union answers and judging for exactly those cases. A pilot folder is frozen to the cases it was prepared with; a later pilot uses a new folder.

In [ ]:
DIAGNOSE = False
if DIAGNOSE:
    done = union_run.prior_completed_case_ids(PRIOR, split=SPLIT, dev_families=DEV_FAMILIES)
    report = union_run.diagnose(PRIOR, done)
    (WORK / ('diagnosis-' + CODE[:8] + '-' + str(len(done)) + '.json')).write_text(json.dumps(report, ensure_ascii=False, indent=1))
    print(json.dumps(report['summary'], ensure_ascii=False, indent=1))
else:
    print('Diagnosis disabled.')

In [ ]:
RUN_PILOT = False          # paid: DeepSeek answers for the finished cases, up to 4 per case
RUN_PILOT_JUDGE = False    # paid: Qwen judging of raw, frozen managed and the 4 configs
PILOT_MAX_CALLS = 200
if RUN_PILOT or RUN_PILOT_JUDGE:
    done = union_run.prior_completed_case_ids(PRIOR, split=SPLIT, dev_families=DEV_FAMILIES)
    PILOT = WORK / ('pilot-union-' + CODE[:8] + '-' + str(len(done)))
    PILOT_QUALITY = WORK / ('pilot-quality-' + CODE[:8] + '-' + str(len(done)))
    if not (PILOT / 'protocol.json').exists():
        union_run.prepare(PRIOR, PILOT, split=SPLIT, dev_families=DEV_FAMILIES, case_ids=done)
    with api_key() as key:
        if RUN_PILOT:
            print(json.dumps(union_run.run(PILOT, HostedGatewayTransport(runner.IDENTITY, key),
                                           max_new_calls=PILOT_MAX_CALLS, workers=4), indent=1))
        if RUN_PILOT_JUDGE:
            if not (PILOT_QUALITY / 'protocol.json').exists():
                print(union_run.export_quality(PILOT, JUDGE_IDENTITY, PILOT_QUALITY, maximum_attempts=2000))
            print(json.dumps(judge.run(PILOT_QUALITY, HostedGatewayTransport(JUDGE_IDENTITY, key),
                                       max_new_calls=PILOT_MAX_CALLS, workers=4), indent=1))
    if (PILOT_QUALITY / 'reviews.jsonl').exists():
        report = union_run.compare(PILOT_QUALITY)
        for arm, dims in report['comparisons'].items():
            print(arm)
            for d, r in dims.items():
                c = r['family_cluster_difference']
                print(f"  {d}: scored {r['scored_pairs']}/{r['planned_pairs']}  win/tie/loss {r['wins']}/{r['ties']}/{r['losses']}  mean {c['mean_difference']}")
else:
    print('Pilot disabled.')

## 1. Prepare the union run (no model call)
Run after section 0 has finished for the dev split. Freezes the family split, the four configs and the code hashes.

In [ ]:
if not (UNION / 'protocol.json').exists():
    protocol = union_run.prepare(PRIOR, UNION, split=SPLIT, dev_families=DEV_FAMILIES)
else:
    protocol = json.loads((UNION / 'protocol.json').read_text())
print(json.dumps({k: protocol[k] for k in ('split', 'case_count', 'maximum_attempts', 'configs')}, indent=1))

## 2. Generate managed answers (paid, DeepSeek)
At most `case_count × 4` answer calls; identical requests share one receipt. Resumable: completed cases are skipped and a sent request is never resent automatically.

In [ ]:
RUN_ANSWERS = False
MAX_CALLS = 120
if RUN_ANSWERS:
    with api_key() as key:
        summary = union_run.run(UNION, HostedGatewayTransport(runner.IDENTITY, key), max_new_calls=MAX_CALLS, workers=4)
    print(json.dumps(summary, indent=1))
else:
    print('Paid answers disabled.', json.dumps(union_run.summarize(UNION), indent=1) if (UNION / 'cases').exists() else '')

## 3. Independent judge (paid, Qwen)
One blinded batch with the raw-RAG answers, the frozen managed answers (both from section 0) and every config, so all conditions share one judge, prompt and session.

Export is a one-time freeze: set `EXPORT_QUALITY = True` only after section 2 shows every case completed or skipped. Exporting earlier is refused.

In [ ]:
JUDGE_IDENTITY = {'provider': 'https://tokendance.space/gateway/v1', 'model': 'qwen3.5-plus',
                  'returned_model': 'qwen3.5-plus-2026-04-20', 'model_family': 'qwen'}
EXPORT_QUALITY = False   # set True once, after section 2 shows every case completed or skipped
if EXPORT_QUALITY and not (QUALITY / 'protocol.json').exists():
    print(union_run.export_quality(UNION, JUDGE_IDENTITY, QUALITY, maximum_attempts=2000))

RUN_JUDGE = False
JUDGE_MAX_CALLS = 200
if RUN_JUDGE and (QUALITY / 'protocol.json').exists():
    with api_key() as key:
        print(json.dumps(judge.run(QUALITY, HostedGatewayTransport(JUDGE_IDENTITY, key), max_new_calls=JUDGE_MAX_CALLS, workers=4), indent=1))
else:
    print('Paid judging disabled.')

## 4. Compare with raw RAG (no model call)
Paired difference per R1–R5, family-clustered bootstrap. On the dev split this is exploratory: use it to choose and freeze configs, not as the reported result.

In [ ]:
if (QUALITY / 'reviews.jsonl').exists():
    report = union_run.compare(QUALITY)
    for arm, dims in report['comparisons'].items():
        print(arm)
        for d, r in dims.items():
            c = r['family_cluster_difference']
            print(f"  {d}: scored {r['scored_pairs']}/{r['planned_pairs']}  win/tie/loss {r['wins']}/{r['ties']}/{r['losses']}  mean {c['mean_difference']}  CI {c['ci95']}")
else:
    print('No judge reviews yet.')

## 5. Test split (only after freezing)
Edit `union_run.DEFAULT_LADDER` in a new commit to keep only the chosen configs, set `SPLIT = 'test'` and a new `CODE`, then run sections 1–4 once. Do not tune again after seeing test results.